<a href="https://colab.research.google.com/github/kshitija2692/Rating-Prediction-via-Prompting/blob/main/API219.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import json
import time

In [ ]:
!pip install openai

OpenRouter Client Setup

In [ ]:
import os
os.environ["OPENROUTER_API_KEY"] = "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"

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

# quick test
resp = client.chat.completions.create(
    model="mistralai/mistral-7b-instruct",
    messages=[{"role": "user", "content": "Say hello in one word"}]
)
print(resp.choices[0].message.content)


 <s> Hello


In [ ]:
df = pd.read_csv('/content/yelp.csv')
df = df[['stars', 'text']]
df = df.sample(200, random_state= 26)
df.head()

,stars,text
8884,1,I hate for my first official review to be a on...
4629,3,NOTE TO SELF/OTHERS: Check zoo hours before go...
726,2,I've really tried to like the Gilbert House. B...
3890,4,had lunch here today outside with the girlfrie...
711,5,It cannot get any better than this my favorite...


Prompt Definitions

In [ ]:
def prompt_1(review):
    return f"""
    Predict the star rating (1–5) for the Yelp review below.
    You must return ONLY valid JSON.
    Do not add any text outside JSON.

    JSON format:
    {{
      "predicted_stars": integer,
      "explanation": string
    }}


    Review: {review}
    """

def prompt_2(review):
    return f"""
    You are a professional sentiment analyst.

    Predict the star rating (1–5) for the Yelp review below.
    You must return ONLY valid JSON.
    Do not add any text outside JSON.

    JSON format:
    {{
      "predicted_stars": integer,
      "explanation": string
    }}

    Review: {review}
    """

def prompt_3(review):
    return f"""
    You are an expert restaurant review analyst.

    Internally analyze sentiment, service, and food quality.
    You must return ONLY valid JSON.
    Do not add any text outside JSON.

    JSON format:
    {{
      "predicted_stars": integer,
      "explanation": string
    }}

    Review: {review}
    """


LLM Call Function

In [ ]:
import json

def call_llm(prompt):
    try:
        response = client.chat.completions.create(
            model="mistralai/mistral-7b-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        text = response.choices[0].message.content.strip()

        # extract JSON safely
        start = text.find("{")
        end = text.rfind("}") + 1
        json_text = text[start:end]

        return json.loads(json_text)

    except Exception as e:
        return None


Normalize Predicted Stars

In [ ]:
def normalize_rating(value):
    try:
        value = float(value)
        return int(round(value))
    except:
        return None

Evaluation Function

In [ ]:
def evaluate_prompt(prompt_func, df):
    total = len(df)
    correct = 0
    valid_json = 0

    for _, row in df.iterrows():
        output = call_llm(prompt_func(row["text"]))

        if output is not None and "predicted_stars" in output:
            predicted = normalize_rating(output["predicted_stars"])

            if predicted is not None:
                valid_json += 1

                if predicted == int(row["stars"]):
                    correct += 1

    return {
        "accuracy": correct / total,
        "json_validity": valid_json / total
    }


Loop Through All 3 Prompts

In [ ]:
prompts = {
    "Prompt 1": prompt_1,
    "Prompt 2": prompt_2,
    "Prompt 3": prompt_3
}

In [ ]:
results = []

for name, prompt_func in prompts.items():
    metrics = evaluate_prompt(prompt_func, df)

    results.append({
        "Prompt": name,
        "Accuracy": metrics["accuracy"],
        "JSON Validity": metrics["json_validity"]
    })

pd.DataFrame(results)


,Prompt,Accuracy,JSON Validity
0,Prompt 1,0.610,0.960
1,Prompt 2,0.605,0.955
2,Prompt 3,0.585,0.845


In [ ]:
results_df = pd.DataFrame(results)
results_df

Reliability / Consistency Check

In [33]:
review = df.iloc[10]["text"]

for i in range(3):
    print(call_llm(prompt_3(review)))


{'predicted_stars': 1, 'explanation': "The review mentions significant issues with cleanliness and food quality, including dirty areas and old, poor-quality ingredients. The negative tone and specific complaints about the restaurant's condition and food freshness justify a 1-star rating."}
{'predicted_stars': 1, 'explanation': 'The review mentions significant issues with cleanliness (dirty lobby, salsa bar, and drink station) and food quality (old guacamole and limp lettuce), which strongly indicate a poor experience. The tone is clearly disappointed, and the reviewer expresses hope for improvement, suggesting a very negative rating.'}
{'predicted_stars': 1, 'explanation': 'The review mentions significant issues with cleanliness and food quality, including a dirty lobby, salsa bar, and drink station, as well as old guacamole and limp lettuce. These negative experiences warrant a 1-star rating.'}
